[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/10_gqa.ipynb)

# 🔴 Hard: Grouped Query Attention (GQA)

Implement **Grouped Query Attention** — used in LLaMA 2, Mistral, etc. to reduce KV cache size.

Like MHA, but with **fewer KV heads** than Q heads. Each group of Q heads shares the same K/V head.

### Signature
```python
class GroupQueryAttention:
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int): ...
    def forward(self, x) -> torch.Tensor:  # self-attention
```

### Requirements
- `self.W_q`: `nn.Linear(d_model, d_model)` — full Q projection
- `self.W_k`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced K projection
- `self.W_v`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced V projection
- `self.W_o`: `nn.Linear(d_model, d_model)` — output projection
- `d_k = d_model // num_heads`
- Expand KV heads with `repeat_interleave` to match Q heads
- When `num_kv_heads == num_heads`, should behave like standard MHA

### my note:
- In GQA, `num_kv_heads <= num_heads`.
- Since `head_dim = d_model // num_heads`, we have `num_kv_heads * head_dim <= d_model`, so the K/V projection output is no larger than `d_model`, more efficient.
- Group size is `num_heads // num_kv_heads`, e.g. with `num_heads = 8` and `num_kv_heads = 2`, GQA forms 2 groups of 4 query heads, with each group sharing one K/V head.

- When `num_kv_heads == num_heads`, each query head has its own key/value head, so it is equivalent to MHA.


In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import torch.nn as nn
import math

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

class GroupQueryAttention:
    def __init__(self, d_model, num_heads, num_kv_heads):
        # pass  # Initialize projections
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.num_kv_heads = num_kv_heads

        self.group_size = self.num_heads // self.num_kv_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, num_kv_heads * self.head_dim)
        self.W_v = nn.Linear(d_model, num_kv_heads * self.head_dim)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x):
        # pass  # Self-attention with grouped KV
        
        batch_size, seq_len_q = x.size(0), x.size(1)
        # seq_len_kv = x.size(1)

        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        
        
        print(Q.shape, K.shape, V.shape)
        
        # [Batch, Seq_len, d_model] -> [Batch, Seq_len, num_heads, head_dim]
        Q_split = Q.view(batch_size, seq_len_q, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        K_split = K.view(batch_size, seq_len_q, self.num_kv_heads, self.head_dim).permute(0, 2, 1, 3)
        V_split = V.view(batch_size, seq_len_q, self.num_kv_heads, self.head_dim).permute(0, 2, 1, 3)
        print('before repeat_interleave: ', Q_split.shape, K_split.shape, V_split.shape)
        K_split = K_split.repeat_interleave(self.group_size, dim=1)
        V_split = V_split.repeat_interleave(self.group_size, dim=1)
        # K_split[N][0~3] will repeat K_split[N][0]
        print('after repeat_interleave: ', Q_split.shape, K_split.shape, V_split.shape)

        multi_head_attn = torch.softmax(Q_split @ K_split.transpose(-2, -1) / math.sqrt(self.head_dim), dim=-1) @ V_split
        print(multi_head_attn.shape) # the seq length of the output is the same as the query sequence length
        attn = multi_head_attn.permute(0, 2, 1, 3).contiguous().view(batch_size, seq_len_q, self.d_model) # concatenate the heads
        print(attn.shape)
        return self.W_o(attn)

In [40]:
# 🧪 Debug
torch.manual_seed(0)
gqa = GroupQueryAttention(d_model=32, num_heads=8, num_kv_heads=2)
print("W_q shape:", gqa.W_q.weight.shape)  # (32, 32)
print("W_k shape:", gqa.W_k.weight.shape)  # (8, 32)  — only 2 KV heads * d_k=4

x = torch.randn(2, 6, 32)
out = gqa.forward(x)
print("Output shape:", out.shape)           # (2, 6, 32)

W_q shape: torch.Size([32, 32])
W_k shape: torch.Size([8, 32])
torch.Size([2, 6, 32]) torch.Size([2, 6, 8]) torch.Size([2, 6, 8])
before repeat_interleave:  torch.Size([2, 8, 6, 4]) torch.Size([2, 2, 6, 4]) torch.Size([2, 2, 6, 4])
tensor([[[-3.3469e-01,  1.2723e+00,  5.2789e-01, -5.0954e-02],
         [-1.3386e+00,  3.6032e-01, -1.1707e+00, -9.4137e-01],
         [ 4.1676e-01, -1.8538e-01, -4.6219e-04,  3.5145e-01],
         [-1.2784e+00, -7.6443e-01, -1.0252e+00,  2.7069e-01],
         [ 1.9798e-02, -4.2088e-02, -4.3578e-01, -2.1603e-01],
         [ 2.6179e-01,  1.9617e-01,  4.6504e-01,  4.0584e-01]],

        [[-1.7059e-01,  2.2739e-02, -1.6250e+00, -1.1711e+00],
         [ 1.1537e+00,  1.1926e-01,  6.6185e-01, -2.5275e-01],
         [-2.8996e-01, -6.9685e-01,  1.1158e+00, -5.4448e-01],
         [-6.9692e-01, -9.8736e-01,  3.0326e-01, -1.2430e-01],
         [-4.9999e-01, -3.3312e-02,  3.8707e-01, -5.8827e-01],
         [-7.6258e-01,  1.2655e-02, -1.1912e+00, -4.4733e-01]]],
       g

In [20]:
from torch_judge import check
check('gqa')


🧪 Testing: Grouped Query Attention (Hard)
──────────────────────────────────────────────────
torch.Size([2, 6, 32]) torch.Size([2, 6, 8]) torch.Size([2, 6, 8])
torch.Size([2, 8, 6, 4]) torch.Size([2, 8, 6, 4]) torch.Size([2, 8, 6, 4])
torch.Size([2, 8, 6, 4])
torch.Size([2, 6, 32])
  ✅ [1/5] Output shape (1.5ms)
  ✅ [2/5] nn.Linear with correct shapes (0.3ms)
torch.Size([1, 4, 16]) torch.Size([1, 4, 16]) torch.Size([1, 4, 16])
torch.Size([1, 4, 4, 4]) torch.Size([1, 4, 4, 4]) torch.Size([1, 4, 4, 4])
torch.Size([1, 4, 4, 4])
torch.Size([1, 4, 16])
  ✅ [3/5] Degenerates to MHA when kv_heads == heads (0.6ms)
  ✅ [4/5] KV heads are shared correctly (0.5ms)
torch.Size([1, 4, 16]) torch.Size([1, 4, 8]) torch.Size([1, 4, 8])
torch.Size([1, 4, 4, 4]) torch.Size([1, 4, 4, 4]) torch.Size([1, 4, 4, 4])
torch.Size([1, 4, 4, 4])
torch.Size([1, 4, 16])
  ✅ [5/5] Gradient flow (1.0ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (3.9ms total)
  Progress saved. Run stat